# Prepare GWAS Conditional SNPs Post-COJO Per Phenotype/Latent

This notebook processes independent signals from GWAS analysis and combines them with pruned SNPs to create phenotype/latent-specific SNP lists.

In [1]:
import pandas as pd
from tqdm import tqdm
from copy import deepcopy
import os

### Configuration

Run only the one you are interested in!

#### DiffAE discovery latents, post r80 selection (nNs_Qntl_INF30_DiffAE128_5Sd_r80_discov_fullDSV3)

In [2]:
# Path to the independent signals TSV file (e.g., gwsig_89snps_post_cojo_GENEanno.tsv)
indep_sigs_path = "/group/glastonbury/GWAS/F20208v3_DiffAE/select_latents_r80/nNs_Qntl_INF30_DiffAE128_5Sd_r80_discov_fullDSV3/nNs_Qntl_INF30_DiffAE128_5Sd_r80_discov_fullDSV3/nNs_Qntl_INF30_DiffAE128_5Sd_r80_discov_fullDSV3/results/analyses/sara/gwsig_89snps_post_cojo_GENEanno.tsv"

# Path to the pruned UKBB SNPs file
snps_file_path = "/group/glastonbury/soumick/PRS/inputs/common/prunedUKBB/plink_maf1p_geno10p_caucasian_prune_250_5_r0p5_ukbb_autosomes_mac100_info0p4.txt"

# Output directory path for latent-wise SNP files
output_path_orig = "/group/glastonbury/soumick/PRS/inputs/F20208v3_DiffAE_select_latents_r80_discov_INF30/lw_gw_indep/lw_SNPs_gwcond_plus_plink_maf1p_geno10p_caucasian_prune_250_5_r0p5_ukbb_autosomes_mac100_info0p4"
output_folder = "/group/glastonbury/soumick/PRS/inputs/F20208v3_DiffAE_select_latents_r80_discov_INF30/lw_gw_indep"

# Column name for phenotype/latent identifier
pheno_col = "Latent"

#### IDP: Aorta (Qntl_WBRIT_6thbasket_aorta_phenos)

In [5]:
indep_sigs_path = "/group/glastonbury/GWAS/CarPheno/6thbasket_aorta_phenos/Qntl_WBRIT_6thbasket_aorta_phenos/Qntl_WBRIT_6thbasket_aorta_phenos/results/analyses/sara/gw_sig_snps_post_cojo.tsv"
snps_file_path = "/group/glastonbury/soumick/PRS/inputs/common/prunedUKBB/plink_maf1p_geno10p_caucasian_prune_250_5_r0p5_ukbb_autosomes_mac100_info0p4.txt" # Same as for the DiffAE latents
output_folder = "/group/glastonbury/soumick/PRS/inputs/F20208_IDPs/lw_gw_indep/aorta"
pheno_col = "latent"

#### IDP: Cardiac (Qntl_WBRIT_6thbasket_cardiac_phenos)


In [2]:
indep_sigs_path = "/group/glastonbury/GWAS/CarPheno/6thbasket_cardiac_phenos/Qntl_WBRIT_6thbasket_cardiac_phenos/Qntl_WBRIT_6thbasket_cardiac_phenos/results/analyses/sara/gw_sig_snps_post_cojo.tsv"
snps_file_path = "/group/glastonbury/soumick/PRS/inputs/common/prunedUKBB/plink_maf1p_geno10p_caucasian_prune_250_5_r0p5_ukbb_autosomes_mac100_info0p4.txt" # Same as for the DiffAE latents
output_folder = "/group/glastonbury/soumick/PRS/inputs/F20208_IDPs/lw_gw_indep/cardiac"
pheno_col = "latent"

### Load Data

In [3]:
# Load independent signals
indep_sigs = pd.read_table(indep_sigs_path)
print(f"Loaded {len(indep_sigs)} independent signals")
print(f"\nColumns: {list(indep_sigs.columns)}")
print(f"\nFirst few rows:")
indep_sigs.head()

Loaded 85 independent signals

Columns: ['SNP', 'Chr', 'bp', 'EA', 'freq', 'b', 'se', 'p', 'n', 'freq_geno', 'bJ', 'bJ_se', 'pJ', 'LD_r', 'latent', 'fullSNPIDs']

First few rows:


,SNP,Chr,bp,EA,freq,b,se,p,n,freq_geno,bJ,bJ_se,pJ,LD_r,latent,fullSNPIDs
0,rs2503715,1,2144107,G,0.872833,-0.069742,0.011388,9.103690e-10,29633.3,0.872833,-0.069742,0.011395,9.321680e-10,0.0,LV_circumferential_strain_global,rs2503715_A_G
1,rs2503715,1,2144107,G,0.872833,-0.055591,0.009094,9.785140e-10,29651.5,0.872833,-0.055591,0.009100,1.001750e-09,0.0,LV_end_systolic_volume,rs2503715_A_G
2,rs55741089,1,11898859,A,0.055493,-0.059637,0.009792,1.126260e-09,31087.7,0.055493,-0.059637,0.009798,1.151380e-09,0.0,LV_myocardial_mass,rs55741089_G_A
3,rs4661656,1,16145970,G,0.463662,0.049333,0.007315,1.536350e-11,31960.4,0.463662,0.049333,0.007320,1.586750e-11,0.0,LV_radial_strain_global,rs4661656_A_G
4,rs143726939,1,16323565,TGTGAGGACTAA,0.482950,-0.063411,0.007581,6.040040e-17,31278.6,0.482950,-0.063411,0.007589,6.530220e-17,0.0,LV_ejection_fraction,rs143726939_T_TGTGAGGACTAA


In [9]:
indep_sigs.latent.unique()

array(['LV_circumferential_strain_global', 'LV_end_systolic_volume',
       'LV_myocardial_mass', 'LV_radial_strain_global',
       'LV_ejection_fraction', 'LV_longitudinal_strain_global',
       'RV_end_systolic_volume', 'RV_ejection_fraction',
       'LV_longitudinal_strain_Segment_5', 'RV_end_diastolic_volume',
       'LV_end_diastolic_volume',
       'LV_mean_myocardial_wall_thickness_global', 'RA_maximum_volume',
       'LV_longitudinal_strain_Segment_3',
       'LV_longitudinal_strain_Segment_6', 'RA_ejection_fraction',
       'LA_minimum_volume', 'RV_stroke_volume', 'LA_ejection_fraction'],
      dtype=object)

In [4]:
# Load pruned SNPs
with open(snps_file_path, "r") as f:
    snps = f.read().splitlines()
    
print(f"Loaded {len(snps)} pruned SNPs")
print(f"\nFirst 5 SNPs: {snps[:5]}")

Loaded 1281751 pruned SNPs

First 5 SNPs: ['rs199745162_A_C', 'rs533090414_C_G', 'rs547227933_C_T', 'rs181431124_A_C', 'rs181028663_A_T']


### Process Each Phenotype/Latent

For each phenotype/latent, combine the pruned SNPs with the phenotype-specific independent signals:

In [5]:
# Get unique phenotypes/latents
phenos = indep_sigs[pheno_col].unique()
print(f"Found {len(phenos)} unique {pheno_col}s")
print(f"\n{pheno_col}s: {sorted(phenos)}")

Found 19 unique latents

latents: ['LA_ejection_fraction', 'LA_minimum_volume', 'LV_circumferential_strain_global', 'LV_ejection_fraction', 'LV_end_diastolic_volume', 'LV_end_systolic_volume', 'LV_longitudinal_strain_Segment_3', 'LV_longitudinal_strain_Segment_5', 'LV_longitudinal_strain_Segment_6', 'LV_longitudinal_strain_global', 'LV_mean_myocardial_wall_thickness_global', 'LV_myocardial_mass', 'LV_radial_strain_global', 'RA_ejection_fraction', 'RA_maximum_volume', 'RV_ejection_fraction', 'RV_end_diastolic_volume', 'RV_end_systolic_volume', 'RV_stroke_volume']


In [18]:
output_path = f"{output_folder}/lw_SNPs_gwcond_plus_{os.path.basename(snps_file_path).replace('.txt','')}"
os.makedirs(output_path, exist_ok=True)
print(f"Output folder created: {output_path}")

Output folder created: /group/glastonbury/soumick/PRS/inputs/F20208_IDPs/lw_gw_indep/cardiac/lw_SNPs_gwcond_plus_plink_maf1p_geno10p_caucasian_prune_250_5_r0p5_ukbb_autosomes_mac100_info0p4


In [19]:
# Process each phenotype/latent
for pheno in tqdm(phenos):
    # Start with a copy of the pruned SNPs
    pheno_snps = deepcopy(snps)
    
    # Add phenotype-specific independent signals
    pheno_snps += indep_sigs[indep_sigs[pheno_col] == pheno].fullSNPIDs.tolist()
    
    # Remove duplicates
    pheno_snps = list(set(pheno_snps))
    
    # Write to file
    output_file = f"{output_path}/{pheno}.txt"
    with open(output_file, "w") as f:
        for snp in pheno_snps:
            f.write(snp + "\n")
    
print(f"\nProcessing complete! Files written to: {output_path}")

100%|██████████| 19/19 [00:18<00:00,  1.04it/s]


Processing complete! Files written to: /group/glastonbury/soumick/PRS/inputs/F20208_IDPs/lw_gw_indep/cardiac/lw_SNPs_gwcond_plus_plink_maf1p_geno10p_caucasian_prune_250_5_r0p5_ukbb_autosomes_mac100_info0p4
